In [ ]:
# %% [markdown]
# # 🎯 CTDE Leader-Follower MAS Event-Triggered Consensus
# 
# **CTDE 架构（Centralized Training Decentralized Execution）**
# - **Actor**: 分散式，只使用本地观测
# - **Critic**: 集中式，使用全局状态 + 联合动作
# - **随机初始化**: 每个 episode 随机化领导者动力学和跟随者初始状态

# %%
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

from mas_cc.config import print_config, NUM_FOLLOWERS, NUM_PINNED
print_config()

# %% [markdown]
# ## 🎲 Randomization Settings
# 
# 每个 episode 随机初始化领导者动力学和跟随者状态，提高泛化能力

# %%
from mas_cc.config import (
    RANDOMIZE_LEADER, RANDOMIZE_FOLLOWER,
    LEADER_AMPLITUDE_RANGE, LEADER_OMEGA_RANGE, LEADER_PHASE_RANGE,
    FOLLOWER_INIT_POS_STD_RANGE, FOLLOWER_INIT_VEL_STD_RANGE,
    LEADER_AMPLITUDE, LEADER_OMEGA, LEADER_PHASE,
    FOLLOWER_INIT_POS_STD, FOLLOWER_INIT_VEL_STD
)

print("🎲 Randomization Configuration:")
print("=" * 50)
print(f"\n📍 Leader Dynamics Randomization: {'✅ Enabled' if RANDOMIZE_LEADER else '❌ Disabled'}")
if RANDOMIZE_LEADER:
    print(f"   Amplitude range: {LEADER_AMPLITUDE_RANGE} (base: {LEADER_AMPLITUDE})")
    print(f"   Omega range: {LEADER_OMEGA_RANGE} (base: {LEADER_OMEGA})")
    print(f"   Phase range: [{LEADER_PHASE_RANGE[0]:.2f}, {LEADER_PHASE_RANGE[1]:.2f}] rad")
else:
    print(f"   Fixed amplitude: {LEADER_AMPLITUDE}")
    print(f"   Fixed omega: {LEADER_OMEGA}")
    print(f"   Fixed phase: {LEADER_PHASE}")

print(f"\n📍 Follower Init Randomization: {'✅ Enabled' if RANDOMIZE_FOLLOWER else '❌ Disabled'}")
if RANDOMIZE_FOLLOWER:
    print(f"   Position std range: {FOLLOWER_INIT_POS_STD_RANGE}")
    print(f"   Velocity std range: {FOLLOWER_INIT_VEL_STD_RANGE}")
else:
    print(f"   Fixed position std: {FOLLOWER_INIT_POS_STD}")
    print(f"   Fixed velocity std: {FOLLOWER_INIT_VEL_STD}")

# %% [markdown]
# ## 📡 Communication Topology (Simulation Only)
# 
# **注意**：拓扑仅用于**模拟通信范围**，神经网络不使用拓扑结构！

# %%
from mas_cc.topology import CommunicationTopology

topology = CommunicationTopology(NUM_FOLLOWERS, num_pinned=NUM_PINNED)
topology.visualize()

# %% [markdown]
# ## 🔍 Environment Test - Verify Randomization

# %%
from mas_cc.environment import BatchedModelFreeEnv
import torch

# 测试随机初始化
test_env = BatchedModelFreeEnv(topology, num_envs=4)

print("🧪 Testing Randomization (4 parallel environments):")
print("=" * 60)

for test_round in range(3):
    _ = test_env.reset()
    
    print(f"\n📍 Reset #{test_round + 1}:")
    print(f"   Leader Amplitudes: {test_env.leader_amplitude.cpu().numpy().round(3)}")
    print(f"   Leader Omegas: {test_env.leader_omega.cpu().numpy().round(3)}")
    print(f"   Leader Phases: {test_env.leader_phase.cpu().numpy().round(3)}")
    print(f"   Follower Pos Std: {test_env.positions[:, 1:].std(dim=1).cpu().numpy().round(3)}")

# %% [markdown]
# ## 🚀 CTDE Training
# 
# **关键特性：**
# - ✅ 每个 episode 随机化领导者动力学
# - ✅ 每个 episode 随机化跟随者初始状态
# - ✅ Actor 只用本地观测（分散执行）
# - ✅ Critic 用全局状态（集中训练）

# %%
from train import train
from mas_cc.config import NUM_EPISODES, VIS_INTERVAL, TRAIN_FOLLOWER_SCALES, SAVE_MODEL_PATH

# 多规模 MAPPO：每个 block 覆盖一个 N（TRAIN_FOLLOWER_SCALES = (10, 20, 30)）。
# 固定规模回退：train(train_follower_scales=(20,))；MASAC replay 只支持单一 N。
trained_agent, topology, dashboard = train(
    num_episodes=NUM_EPISODES,
    vis_interval=VIS_INTERVAL,
    show_dashboard=True,
    train_follower_scales=TRAIN_FOLLOWER_SCALES,
)
print(f"Checkpoint: {SAVE_MODEL_PATH}")

# %% [markdown]
# ## 📊 Evaluation on Different Scenarios
# 
# 评估智能体在**不同领导者动力学**下的泛化能力

# %%
from mas_cc.utils import plot_evaluation, collect_trajectory, evaluate_agent
from mas_cc.environment import ModelFreeEnv
import matplotlib.pyplot as plt
import numpy as np

# 标准评估
print("📊 Standard Evaluation (Randomized Scenarios):")
print("=" * 50)
from mas_cc.config import EVAL_NUM_TESTS, EVAL_SAVE_PATH
plot_evaluation(trained_agent, topology, num_tests=EVAL_NUM_TESTS, save_path=EVAL_SAVE_PATH)

# %% [markdown]
# ## 🌐 Cross-Scale Generalization — 同一 Actor 跑不同 N
#
# v2 观测固定为 `10 + 7K`（默认 K=4 → 38 维），与 follower 数 N 无关；
# `ActorOnlyPolicy.from_checkpoint` 直接恢复共享 Actor，无需完整 MAPPO/SAC agent。

# %%
from mas_cc.agent import ActorOnlyPolicy
from mas_cc.config import STATE_DIM, TOP_K, SAVE_MODEL_PATH as CKPT_PATH

policy = ActorOnlyPolicy.from_checkpoint(CKPT_PATH)
print(f"v2 obs: STATE_DIM={STATE_DIM} = 10 + 7 × K({TOP_K})")
print(f"Loaded actor-only policy from {CKPT_PATH}")

for n in (10, 20, 30):
    scale_topology = CommunicationTopology(n)
    scale_env = ModelFreeEnv(scale_topology)
    scale_metrics = evaluate_agent(policy, scale_env, num_episodes=2)
    print(f"N={n}: mean_tracking_error={scale_metrics['mean_tracking_error']:.4f}, "
          f"mean_comm_rate={scale_metrics['mean_comm_rate']*100:.1f}%")

# %% [markdown]
# ## 🔬 Generalization Test - Different Leader Dynamics

# %%
# 测试不同领导者动力学下的性能
def test_specific_dynamics(agent, topology, amplitude, omega, num_steps=None, phase=None, traj_type=None):
    """测试特定领导者动力学下的性能（默认值统一来自 config）"""
    from mas_cc.environment import ModelFreeEnv
    from mas_cc.config import LEADER_PHASE, LEADER_TRAJECTORY_TYPES, GENERALIZATION_TEST_STEPS
    
    if num_steps is None:
        num_steps = int(GENERALIZATION_TEST_STEPS)
    if phase is None:
        phase = float(LEADER_PHASE)
    if traj_type is None:
        traj_type = LEADER_TRAJECTORY_TYPES[0] if LEADER_TRAJECTORY_TYPES else 'sine'

    # 先安装固定场景再 reset：第 0 步动作就使用指定轨迹类型/领导者参数。
    scenario = {
        'leader_id': 0,
        'adj_matrix': topology.adj,
        'pinned_ids': [int(i) for i in topology.pinned_ids],
        'trajectory_type': traj_type,
        'leader_amplitude': float(amplitude),
        'leader_omega': float(omega),
        'leader_phase': float(phase),
        'initial_positions': np.zeros(topology.num_followers),
        'initial_velocities': np.zeros(topology.num_followers),
    }
    env = ModelFreeEnv(topology, auto_reset=False)
    state = env.reset(scenario=scenario)
    total_reward = 0
    total_error = 0
    total_comm = 0
    
    times = [0]
    leader_pos = [env.positions[0].item()]
    follower_pos = [env.positions[1:].cpu().numpy().copy()]
    
    for step in range(num_steps):
        action = agent.select_action(state, deterministic=True)
        state, reward, _, info = env.step(action)
        
        total_reward += reward
        total_error += info['tracking_error']
        total_comm += info['comm_rate']
        
        times.append(env.t)
        leader_pos.append(env.positions[0].item())
        follower_pos.append(env.positions[1:].cpu().numpy().copy())
    
    return {
        'reward': total_reward,
        'avg_error': total_error / num_steps,
        'avg_comm': total_comm / num_steps,
        'times': np.array(times),
        'leader_pos': np.array(leader_pos),
        'follower_pos': np.array(follower_pos)
    }

# 测试不同场景（全部由 config 派生，避免写死）
from mas_cc.config import (
    LEADER_AMPLITUDE_RANGE, LEADER_OMEGA_RANGE,
    LEADER_AMPLITUDE, LEADER_OMEGA,
    GENERALIZATION_INCLUDE_OOD, GENERALIZATION_OOD_AMPLITUDE, GENERALIZATION_OOD_OMEGA
)
a_lo, a_hi = LEADER_AMPLITUDE_RANGE
w_lo, w_hi = LEADER_OMEGA_RANGE

test_scenarios = [
    {'name': 'Range Low', 'amplitude': a_lo, 'omega': w_lo},
    {'name': 'Base', 'amplitude': LEADER_AMPLITUDE, 'omega': LEADER_OMEGA},
    {'name': 'Range High', 'amplitude': a_hi, 'omega': w_hi},
]
if GENERALIZATION_INCLUDE_OOD:
    test_scenarios.append({'name': 'OOD Omega', 'amplitude': GENERALIZATION_OOD_AMPLITUDE, 'omega': GENERALIZATION_OOD_OMEGA})

print("\n🔬 Generalization Test - Different Leader Dynamics:")
print("=" * 70)
print(f"{'Scenario':<15} | {'Amplitude':>9} | {'Omega':>7} | {'Reward':>10} | {'Avg Error':>10} | {'Comm Rate':>9}")
print("-" * 70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, scenario in enumerate(test_scenarios):
    result = test_specific_dynamics(
        trained_agent, topology, 
        scenario['amplitude'], scenario['omega']
    )
    
    print(f"{scenario['name']:<15} | {scenario['amplitude']:>9.1f} | {scenario['omega']:>7.2f} | "
          f"{result['reward']:>10.2f} | {result['avg_error']:>10.4f} | {result['avg_comm']*100:>8.1f}%")
    
    # 绘图
    ax = axes[idx]
    ax.plot(result['times'], result['leader_pos'], 'r-', lw=2.5, label='Leader')
    
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, result['follower_pos'].shape[1]))
    for i in range(min(4, result['follower_pos'].shape[1])):
        ax.plot(result['times'], result['follower_pos'][:, i], color=colors[i], alpha=0.7, lw=1)
    
    avg_follower = result['follower_pos'].mean(axis=1)
    ax.plot(result['times'], avg_follower, 'g--', lw=2, label='Avg Follower')
    
    ax.set_title(f"{scenario['name']} (A={scenario['amplitude']}, ω={scenario['omega']})\n"
                f"Error: {result['avg_error']:.4f}, Comm: {result['avg_comm']*100:.1f}%", fontsize=10)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Position')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
from mas_cc.config import GENERALIZATION_SAVE_PATH
plt.savefig(GENERALIZATION_SAVE_PATH, dpi=150, bbox_inches='tight')
plt.show()

print("-" * 70)
print("✅ Generalization test complete!")

# %% [markdown]
# ## 📈 CTDE State Space Analysis

# %%
from mas_cc.config import (
    STATE_DIM, SELF_STATE_DIM, SELF_ROLE_DIM, LOCAL_OBS_DIM,
    MAX_NEIGHBORS, NEIGHBOR_FEAT_DIM,
    GLOBAL_STATE_DIM, NUM_AGENTS, ACTION_DIM, NUM_FOLLOWERS,
    OBS_VERSION, OBS_LAYOUT,
)

print("📊 CTDE State Space Analysis (v2 observation):")
print("=" * 60)
print(f"   OBS_VERSION={OBS_VERSION}, layout={OBS_LAYOUT}")
print(f"   Actor input: STATE_DIM={STATE_DIM} = 10 + 7 × K({MAX_NEIGHBORS})")
print(f"   (默认 K=4 → 38 维；维度与 N 无关，支持跨规模共享 Actor)")
print(f"     - Self {SELF_STATE_DIM} = local {LOCAL_OBS_DIM} (relative coords: 自身相对 leader 广播/估计) + role {SELF_ROLE_DIM}")
print(f"     - Each of K={MAX_NEIGHBORS} neighbor slots: {NEIGHBOR_FEAT_DIM} 维")
print(f"       = 6 visible features (relative position/velocity, relative broadcast/estimate, age, has_leader_est) + explicit valid bit")
print(f"     - Attention over slots is permutation-invariant; invalid slots are masked by valid=0")
print(f"   Output: Action ({ACTION_DIM} dim) = velocity change + communication threshold")

print("\n🎯 MAPPO Value (centralized training):")
print(f"   Input: DeepSets global state ({GLOBAL_STATE_DIM} dim, permutation-invariant)")
print(f"   - 只吃全局状态，不拼接联合动作；与 Actor 的 N 无关")
print(f"   Output: state value (1 dim)")

print("\n🎯 SAC Critic (centralized training):")
print(f"   Input: global state ({GLOBAL_STATE_DIM} dim) + joint action ({NUM_FOLLOWERS * ACTION_DIM} dim)")
print(f"   - 只有 SAC Critic 使用联合动作；不要把它写成通用 Critic 输入")
print(f"   Output: Q-value (1 dim)")

print("\n🔑 Key CTDE Properties:")
print("   ✅ Actor does NOT access global state during execution")
print("   ✅ MAPPO Value / SAC Critic have full observability during training")
print("   ✅ Each agent uses shared policy (parameter sharing)")
print("   ✅ v2 obs 用相对坐标 + 显式 valid / has_leader_est，泛化到不同 N")

# %% [markdown]
# ## 💾 Save Results

# %%
# 保存训练摘要
if dashboard:
    summary = dashboard.get_summary()
    print("\n📊 Training Summary:")
    print(f"   Best Reward: {summary['best_reward']:.2f}")
    print(f"   Final Reward: {summary['final_reward']:.2f}")
    print(f"   Final Tracking Error: {summary['final_tracking_error']:.4f}")
    print(f"   Final Comm Rate: {summary['final_comm_rate']*100:.1f}%")
    print(f"   Total Episodes: {summary['total_episodes']}")
    print(f"   Elapsed Time: {summary['elapsed_time']:.1f}s")

# 保存模型信息
from mas_cc.config import SAVE_MODEL_PATH
print(f"\n💾 Model saved to: {SAVE_MODEL_PATH}")